# Testing the Saved Model

This notebook loads the model saved by `train.ipynb` and evaluates it on the test set.
Nothing is trained here: all imputers, encoders and scalers come from the saved file and are only used with `transform`.

## 1. Import Libraries

In [10]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load the Saved Model and Preprocessing Objects

In [11]:
bundle = joblib.load('model_file.joblib')
model = bundle['model']

print("Final model:", bundle['model_name'])
print("Saved objects:", list(bundle.keys()))

Final model: Random Forest Regressor
Saved objects: ['model', 'model_name', 'numeric_imputer', 'categorical_imputer', 'severity_encoder', 'gender_encoder', 'onehot_encoder', 'scaler', 'target_scaler', 'uses_target_scaling', 'numeric_cols', 'categorical_cols', 'nominal_cols', 'input_columns', 'feature_columns', 'domain_bounds', 'test_size', 'random_state', 'train_metrics']


## 3. Load the Data

In [12]:
data = pd.read_csv('test.csv')
print("Shape:", data.shape)


Shape: (20000, 15)


## 4. Apply the Same Cleaning as in Training

Invalid values are turned into nulls with the same bounds that were used in `train.ipynb`.

In [13]:
for col, (lo, hi) in bundle['domain_bounds'].items():
    invalid = ((data[col] < lo) | (data[col] > hi)) & data[col].notna()
    data.loc[invalid, col] = np.nan

print("Nulls after cleaning:", data.isnull().sum().sum())

Nulls after cleaning: 6904


## 5. Prepare the Test Data


Same `test_size` and `random_state` as in `train.ipynb` (both are read from the saved file).
Only the test part is kept.

In [14]:
model_data = data.drop(columns=['Patient_ID', 'Doctor_ID', 'Admission_Date', 'Discharge_Date'])

target_column = 'Treatment_Cost_USD'
X_test = model_data.drop(columns=[target_column])
y_test = model_data[target_column]

print("Test:", X_test.shape)
print("Same columns as in training:", list(X_test.columns) == bundle['input_columns'])


Test: (20000, 10)
Same columns as in training: True


## 6. Apply the Saved Preprocessing (transform only, no fitting)

In [15]:
num_cols = bundle['numeric_cols']
cat_cols = bundle['categorical_cols']
nom_cols = bundle['nominal_cols']

# missing values
X_test[num_cols] = bundle['numeric_imputer'].transform(X_test[num_cols])
X_test[cat_cols] = bundle['categorical_imputer'].transform(X_test[cat_cols])

# encoding
X_test[['Severity_Level']] = bundle['severity_encoder'].transform(X_test[['Severity_Level']])
X_test['Gender'] = bundle['gender_encoder'].transform(X_test['Gender'])

onehot = bundle['onehot_encoder']
test_onehot = pd.DataFrame(onehot.transform(X_test[nom_cols]),
                           columns=onehot.get_feature_names_out(nom_cols), index=X_test.index)
X_test = pd.concat([X_test.drop(columns=nom_cols), test_onehot], axis=1)

# scaling
X_test[num_cols] = bundle['scaler'].transform(X_test[num_cols])

# same column order as in training
X_test = X_test[bundle['feature_columns']]
print("Nulls left:", X_test.isnull().sum().sum())
print("Features:", X_test.shape[1])

Nulls left: 0
Features: 30


## 7. Predict and Evaluate

In [16]:
predictions = model.predict(X_test)
if bundle['uses_target_scaling']:
    predictions = bundle['target_scaler'].inverse_transform(
        predictions.reshape(-1, 1)
    ).ravel()

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Model:", bundle['model_name'])
print("MAE:", f"{mae:,.2f}")
print("MSE:", f"{mse:,.2f}")
print("R2: ", f"{r2:.4f}")


Model: Random Forest Regressor
MAE: 1,426.32
MSE: 8,087,645.80
R2:  0.8626


## 8. Actual vs Predicted Treatment Cost

In [17]:
comparison = pd.DataFrame({
    'Actual_Treatment_Cost_USD': y_test.values,
    'Predicted_Treatment_Cost_USD': predictions.round(2),
})
comparison['Absolute_Error'] = (comparison['Actual_Treatment_Cost_USD'] - comparison['Predicted_Treatment_Cost_USD']).abs().round(2)

comparison.head(20)

,Actual_Treatment_Cost_USD,Predicted_Treatment_Cost_USD,Absolute_Error
0,3172.0,3028.91,143.09
1,4280.0,3983.31,296.69
2,584.0,606.47,22.47
3,3794.0,3017.23,776.77
4,377.0,557.78,180.78
5,448.0,615.84,167.84
6,148.0,551.68,403.68
7,746.0,616.30,129.70
8,1836.0,3075.87,1239.87
9,668.0,611.77,56.23
